# Multiple regression 
## Introduction


## Definitions
### Multiple linear regression
### Key terminology
### The mathematical model
#### Estimation
#### Interpreting the coefficients
#### Example


## Assumptions
### Linearity
### Independence of errors
### Homoscedasticity
### Normality of errors
### Multicollinearity


## Real-world example
### Getting the data


In [ ]:
import pandas as pd

# Load the dataset from the URL of the MOOC
data = pd.read_csv(
    # "https://lms.fun-mooc.fr/c4x/Paris11/15001/asset/smp2.csv",
    "./data/smp2.csv",  # If workding from the GitHub repository
    delimiter=';',  # Specify the delimiter as ';'
)

print("First rows of the dataframe:")
print(data.head())

In [ ]:
print("Dataset information:")
print(data.info())

In [ ]:
print("Descriptive statistics:")
print(data.describe().round(2))

In [ ]:
print("Descriptive statistics on selection of variables:")
print(data[['age', 'dep.cons', 'scz.cons', 'juge.enfant']].describe())

### Building the regression model 


In [ ]:
import statsmodels.api as sm

# Create a new DataFrame with only the relevant variables
data_regression = data[
    ['dur.interv', 'age', 'dep.cons', 'subst.cons', 'scz.cons']].copy()

# Drop rows with missing values in any of the selected columns
data_regression = data_regression.dropna()

# Define the dependent variable (Y)
y = data_regression['dur.interv']

# Define the independent variables (X)
X = data_regression[['age', 'dep.cons', 'subst.cons', 'scz.cons']]

# Add a constant term to the independent variables matrix for the intercept
X = sm.add_constant(X)

# Create and fit the OLS model
model_sm = sm.OLS(y, X)
result_sm = model_sm.fit()

# Print the regression result summary
print(result_sm.summary())

In [ ]:
import statsmodels.formula.api as smf
from statsmodels.stats.stattools import durbin_watson

# Define the model formula using patsy syntax
formula = """
Q('dur.interv') ~ age + Q('dep.cons') + Q('subst.cons') + Q('scz.cons')
"""

# Fit the OLS model using the formula and data
model = smf.ols(formula, data=data_regression)

# Fit the model
result = model.fit()

# Print the regression result with a slightly different method
print(result.summary2())

In [ ]:
import pingouin as pg

# Fit the linear regression model using Pingouin
result_pingouin = pg.linear_regression(X, y)

# Print the regression result summary
print("Result of the multiple regression (Pingouin):")
print(result_pingouin.round(2))

### Interpretation of the regression report 
#### Overall model fit
#### Coefficients
#### Other diagnostics


In [ ]:

import matplotlib.pyplot as plt
import seaborn as sns

# Get the residuals from the model
residuals = result.resid

# Create subplots for the histogram and Q-Q plot
fig, axes = plt.subplots(1, 2, figsize=(6, 3))

# Plot the histogram of residuals
sns.histplot(residuals, kde=True, ax=axes[0]).lines[0].set_color('crimson')
axes[0].set_xlabel("Residuals")
axes[0].set_ylabel("Frequency")
axes[0].set_title("Histogram of residuals")

# Plot the Q-Q plot of residuals
sm.qqplot(residuals, line='45', fit=True, ax=axes[1])
axes[1].set_title("Q-Q plot of residuals")

plt.tight_layout();

### Visualizing regression relationships 
#### Partial regression plot


In [ ]:

# Create the partial regression plot grid
fig = plt.figure(figsize=(6, 4))
sm.graphics.plot_partregress_grid(result, fig=fig, grid=(2,3))
plt.tight_layout();

#### Visualizing regression results for one regressor


In [ ]:

# Create the partial regression plot grid
fig = plt.figure(figsize=(6,6))
sm.graphics.plot_regress_exog(result, 'age', fig=fig)
plt.tight_layout();

### Visualizing best-fit line


In [ ]:

# Calculate predicted values
data_regression['pred'] = result.predict(X)

# Plot predicted vs. actual Y values
plt.figure(figsize=(3.5, 3.25))
sns.regplot(
    x='dur.interv', y='pred', data=data_regression,
    ci=None, marker='+', color='brown',
    x_jitter=3, y_jitter=2)

# Add a 45-degree line
plt.plot(
    [data_regression['dur.interv'].min(),
     data_regression['dur.interv'].max()],
    [data_regression['dur.interv'].min(),
     data_regression['dur.interv'].max()],
    color='gray', linestyle='--', label='Perfect prediction')

plt.xlabel("Actual dur.interv")
plt.ylabel("Predicted dur.interv")
plt.xlim((0, 120))
plt.ylim((0, 120))
plt.title("Actual vs. predicted")
plt.legend();

### R² and adjusted R²


In [ ]:
# Get the number of observations from the fitted model
n = result.nobs

# Get the degrees of freedom of the residuals
df_residuals = result.df_resid

# Calculate adjusted R-squared manually
adjusted_r_squared = 1 - (1 - result.rsquared) * (n - 1) / df_residuals

# Print adjusted R-squared from the manual calculation and from Statsmodels
print(f"Adjusted R-squared (manual) = {adjusted_r_squared:.4f}")
print(f"Adjusted R-squared (statsmodels) = {result.rsquared_adj:.4f}")

## Advanced techniques
### Improvement of the fit


In [ ]:
# List of variables to include in the second model
variables_overfit = [
    'age',
    'n.enfant',
    'grav.cons',
    'dep.cons',
    'ago.cons',
    'alc.cons',
    'subst.cons',
    'scz.cons']

# Create a new DataFrame with only the relevant variables
data_overfit = data[['dur.interv'] + variables_overfit].copy()

# Drop rows with missing values in any of the selected columns
data_overfit = data_overfit.dropna()

# Define the dependent variable (Y)
y_overfit = data_overfit['dur.interv']

# Define the independent variables (X)
X_overfit = data_overfit[variables_overfit]

# Add a constant term to the independent variables matrix for the intercept
X_overfit = sm.add_constant(X_overfit)

# Create and fit the OLS model
model_overfit = sm.OLS(y_overfit, X_overfit)
result_overfit = model_overfit.fit()

# Print the regression result summary
print("Summary of the multiple regression on the data with \
additional variables:")
print(result_overfit.summary().tables[0])

### Variable selection


### Collinearity and multicollinearity 


### Three-dimensional visualization 


In [ ]:
import numpy as np

# Create a new DataFrame with only the relevant variables
data_hyperplane = data[['dur.interv', 'age', 'subst.cons']].copy()

# Drop rows with missing values in any of the selected columns
data_hyperplane = data_hyperplane.dropna()

# Define the model formula using patsy syntax
formula_hyperplane = "Q('dur.interv') ~ age + Q('subst.cons')"

# Fit the OLS model using the formula and data
model_hyperplane = smf.ols(formula_hyperplane, data=data_hyperplane)
result_hyperplane = model_hyperplane.fit()

# Print the regression result summary
print("Result of the regression on the hyperplane data:")
print(result_hyperplane.summary2().tables[1].round(3))

In [ ]:

# Create the 3D plot
fig = plt.figure(figsize=(6, 3.5))
ax = fig.add_subplot(111, projection='3d')

# Scatter plot of the data points
ax.scatter(
    data_hyperplane['age'],
    data_hyperplane['subst.cons'],
    data_hyperplane['dur.interv'],
    c='blue', marker='o', alpha=0.75)

# Create the hyperplane
x_surf, y_surf = np.meshgrid(
    np.linspace(
        data_hyperplane['age'].min(),
        data_hyperplane['age'].max(), 10),
    np.linspace(
        data_hyperplane['subst.cons'].min(),
        data_hyperplane['subst.cons'].max(), 10))
exog = pd.DataFrame({'age': x_surf.ravel(), 'subst.cons': y_surf.ravel()})
out = result_hyperplane.predict(exog=exog)
z_surf = out.values.reshape(x_surf.shape)
ax.plot_surface(x_surf, y_surf, z_surf, color='red', alpha=0.5)

# Set axis labels
ax.set_xlabel("age")
ax.set_ylabel("subst.cons")
ax.set_zlabel("dur.interv")
ax.set_title("Hyperplane of dur.interv ~ age + subst.cons")

# Set the view angle
ax.view_init(elev=10, azim=325);

### Interactions


In [ ]:
# List of variables to include in the model
variables_interaction = ['age', 'dep.cons', 'subst.cons', 'scz.cons']

# Create a new DataFrame with only the relevant variables
data_interaction = data[['dur.interv'] + variables_interaction].copy()

# Drop rows with missing values in any of the selected columns
data_interaction = data_interaction.dropna()

# Create the interaction term, it's a simple mathematical multiplication
data_interaction['interaction'] = (
    data_interaction['dep.cons'] * data_interaction['subst.cons'])

# Define the dependent variable (Y)
y_interaction = data_interaction['dur.interv']

# Define the independent variables (X), including the interaction term
X_interaction = data_interaction[variables_interaction + ['interaction']]

# Add a constant term to the independent variables matrix for the intercept
X_interaction = sm.add_constant(X_interaction)

# Create and fit the OLS model
model_interaction = sm.OLS(y_interaction, X_interaction)
result_interaction = model_interaction.fit()

# Print the regression result summary
print("Result of the regression with interaction term \
(using multiplication):")
print(result_interaction.summary2().tables[1].round(3))

In [ ]:
# Define the model formula with interaction using '*'
formula_asterisk = """
Q('dur.interv') ~ age + Q('scz.cons') + Q('dep.cons') * Q('subst.cons')
"""
#"""Q('dur.interv') ~ age
#+ Q('scz.cons') + Q('dep.cons') + Q('subst.cons')
#+ Q('dep.cons') : Q('subst.cons')"""

# Fit the model
model_asterisk = smf.ols(formula_asterisk, data=data_interaction)
result_asterisk = model_asterisk.fit()

# Print the results
print("Result of the regression with interaction term \
(using asterisk-notation):")
print(result_asterisk.summary2().tables[1].round(3))

In [ ]:
# Define the model formula with interaction using ':'
formula_colon = """
Q('dur.interv') ~ age + Q('scz.cons') + Q('dep.cons'):Q('subst.cons')
"""

# Fit the model
model_colon = smf.ols(formula_colon, data=data_interaction)
result_colon = model_colon.fit()

# Print the results
print("Result of the regression with interaction term \
(using colon-notation):")
print(result_colon.summary2().tables[1].round(3))

### Quadratic and higher-order polynomial terms


In [ ]:
# Define the model formula with the interaction term 
# and quadratic term for 'age'
formula_quadratic = """Q('dur.interv') ~ age 
+ Q('scz.cons') + Q('dep.cons') * Q('subst.cons') 
+ I(age ** 2)"""

# Fit the OLS model using the formula and data
model_quadratic = smf.ols(formula_quadratic, data=data_interaction)

# Fit the model
result_quadratic = model_quadratic.fit()

# Print the regression result summary
print("Result of the regression with quadratic term:")
print(f"Adjusted R² = {result_quadratic.rsquared_adj:.3f}")
print(result_quadratic.summary2().tables[1].round(3))

### Special variables


In [ ]:
# Define the model formula with the logarithm of 'age'
formula_log = """Q('dur.interv') ~ np.log(age)
+ Q('scz.cons') + Q('dep.cons') * Q('subst.cons')"""

# Fit the OLS model using the formula and data
model_log = smf.ols(formula_log, data=data_interaction)

# Fit the model
result_log = model_log.fit()

# Print the regression result summary
print("Result of the regression with logarithmic term:")
print(f"Adjusted R² = {result_log.rsquared_adj:.3f}")
print(result_log.summary2().tables[1].round(3))

### Removal of intercept 


In [ ]:
# Define the model formula with the intercept removed
formula_no_intercept = """
Q('dur.interv') ~ age + Q('scz.cons') + Q('dep.cons') * Q('subst.cons') - 1
"""

# Fit the OLS model using the formula and data
model_no_intercept = smf.ols(formula_no_intercept, data=data_interaction)

# Fit the model
result_no_intercept = model_no_intercept.fit()

# Print the regression result summary
print("Result of the regression without intercept:")
print(f"Adjusted R² = {result_no_intercept.rsquared_adj:.3f}")
print(result_no_intercept.summary2().tables[1].round(3))

In [ ]:

# Create the partial regression plot grid
fig = plt.figure(figsize=(6, 4))
sm.graphics.plot_partregress_grid(result_no_intercept, fig=fig, grid=(2,3))
plt.tight_layout();

### Categorical variables


In [ ]:
# Examine the unique categories in the 'prof' variable
print(data['prof'].unique())

In [ ]:
# Create a new DataFrame with only the relevant variables
data_categorical = data[
    ['dur.interv', 'age', 'dep.cons', 'subst.cons', 'scz.cons', 'prof']
].copy()

# Drop rows with missing values in any of the selected columns
data_categorical = data_categorical.dropna()

# Define the model formula with 'prof' as a categorical variable
formula_categorical = """Q('dur.interv') ~ age 
+ Q('dep.cons') + Q('subst.cons') + Q('scz.cons') + C(prof)"""

# Fit the OLS model using the formula and data
model_categorical = smf.ols(formula_categorical, data=data_categorical)

# Fit the model
result_categorical = model_categorical.fit()

# Print the regression result summary
print("Result of the regression with categorical variable:")
print(result_categorical.summary2().tables[1].round(3))

In [ ]:
# Define the model formula with 'prof' as a categorical variable 
# and 'ouvrier' as the reference
formula_categorical_releveled = """
Q('dur.interv') ~ age + Q('dep.cons') + Q('subst.cons') + Q('scz.cons') 
+ C(prof, Treatment(reference='ouvrier'))
"""

# Fit the OLS model using the formula and data
model_categorical_releveled = smf.ols(
    formula=formula_categorical_releveled,
    data=data_categorical)

# Fit the model
result_categorical_releveled = model_categorical_releveled.fit()

# Print the regression result summary
print("Result of the regression with categorical variable (releveled):")
print(result_categorical_releveled.summary2().tables[1].round(3))

## Conclusion
